# **NetMHCpan wrapper testing notebook**

In [1]:
import pandas as pd
import numpy as np
import sys, os
sys.path.append(os.path.dirname(os.getcwd()))
from src.single_predictor import SinglePredictor
from src.pan_predictor import PanPredictor
from pathlib import Path
import subprocess
from tqdm import tqdm
import time



### **Testing single predictor class**

In [2]:
PATH_TO_NETMHCPAN = Path('/home/aalarkin/Work/netmhcpan_wrapper/netMHCpan-4.2bstatic.Linux/netMHCpan-4.2')
TMPDIR = Path('/home/aalarkin/Work/netmhcpan_wrapper/tmp')
PATH_TO_SUPERGROUPS = Path('/home/aalarkin/Work/netmhcpan_wrapper/datasets/mhc_supergroups.txt')
PATH_TO_MAPPING = Path('/home/aalarkin/Work/netmhcpan_wrapper/datasets/allele_nomenclature_mapping.txt')


In [3]:
single_predictor = SinglePredictor(
    path_to_netmhcpan=PATH_TO_NETMHCPAN,
    path_to_mapping=PATH_TO_MAPPING,
    tmpdir=TMPDIR,
    n_cores=8,
)

#testing on sample dataset
df = pd.read_csv('../datasets/chowell.txt', sep='\t')[:100]
df = single_predictor.match_hla(df, 'hla')
df = single_predictor.predict_affinity_dataframe(df, 'antigen.epitope', 'matched_hla')
display(df)


,antigen.epitope,hla,immunogenicity,matched_hla,%Rank Score_BA,%Rank_BA,Aff(nM)
0,ITDFNIDTY,HLA-A*01:01,Positive,HLA-A01:01,0.732399,0.008,18.09
1,ATDALMTGF,HLA-A*01:01,Positive,HLA-A01:01,0.427019,0.133,492.51
2,SSNIMSESY,HLA-A*01:01,Positive,HLA-A01:01,0.464926,0.095,326.81
3,VSEKYTDMY,HLA-A*01:01,Positive,HLA-A01:01,0.710627,0.009,22.90
4,FTDWANKQY,HLA-A*01:01,Positive,HLA-A01:01,0.715700,0.009,21.67
...,...,...,...,...,...,...,...
95,IADMGHLKY,HLA-A*01:01,Negative,HLA-A01:01,0.676161,0.011,33.24
96,DSDLRNDSY,HLA-A*01:01,Negative,HLA-A01:01,0.556244,0.040,121.67
97,ITEDKTELY,HLA-A*01:01,Negative,HLA-A01:01,0.718329,0.009,21.07
98,ALQEALTEHY,HLA-A*01:01,Negative,HLA-A01:01,0.259031,0.589,3032.42


In [4]:
#testing on data with inconsistent labels
inconsistent_data = {
    "IVDSLTEMY": "HLA-A*01:01",
    "LIDGIFLRY": "HLA-A*0101",
    "GLLPSLLLLL": "HLA-A0201",
    "EVISVMKRR": "A6801",
    "FQKVITEY": "B*15:01",
    "SEEDFIRSL": "B*4104",
    "VMADRTRHL": "E0103",
    "ANADLEVKI": "HLA-C*0602"
}
df = pd.DataFrame(list(inconsistent_data.items()), columns=['epitope', 'hla'])

df = single_predictor.match_hla(df, 'hla')
df = single_predictor.predict_affinity_dataframe(df, 'epitope', 'matched_hla')
display(df)



,epitope,hla,matched_hla,%Rank Score_BA,%Rank_BA,Aff(nM)
0,IVDSLTEMY,HLA-A*01:01,HLA-A01:01,0.658625,0.013,40.19
1,LIDGIFLRY,HLA-A*0101,HLA-A01:01,0.675019,0.011,33.66
2,GLLPSLLLLL,HLA-A0201,HLA-A02:01,0.752222,0.131,14.60
3,EVISVMKRR,A6801,HLA-A68:01,0.834083,0.041,6.02
4,FQKVITEY,B*15:01,HLA-B15:01,0.563743,0.624,112.19
5,SEEDFIRSL,B*4104,HLA-B41:04,0.578451,0.147,95.69
6,VMADRTRHL,E0103,HLA-E01:03,0.384711,0.032,778.44
7,ANADLEVKI,HLA-C*0602,HLA-C06:02,0.128117,5.632,12501.21


### **Testing pan predictor class**

In [5]:
predictor = PanPredictor(
    path_to_netmhcpan=PATH_TO_NETMHCPAN,
    path_to_supergroups=PATH_TO_SUPERGROUPS,
    tmpdir=TMPDIR,
    n_cores=16,
)

#no superfamily known
epitopes = ['LIDGIFLRY', 'VMADRTRHL', 'ANADLEVKI']
df = pd.DataFrame(epitopes, columns=['epitope'])
df = predictor.pan_hla_matching_prediction(df, 'epitope')
display(df)

#results are different from testing notebook as fixed logics bugs for supergroup matching


,epitope,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,SLA-1-YDL01,0.733159,0.009,17.94
1,VMADRTRHL,BoLA-6:01301,0.657554,0.227,40.66
2,ANADLEVKI,BoLA-3:01101,0.354021,0.418,1085.02


In [6]:
#superfamilies are known

data = {
    "LIDGIFLRY": "HLA-A01", #ref - hla a0101
    "FQKVITEY": "HLA-B15",    #ref - hla b1501
    "VMADRTRHL": "HLA-E01",      #ref - e0103
    "ANADLEVKI": "HLA-C06"       #ref - HLA-C*0602
}

df = pd.DataFrame(list(data.items()), columns=['epitope', 'family'])

df = predictor.pan_hla_matching_prediction(df, 'epitope', 'family')
display(df)


,epitope,family,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,HLA-A01,HLA-A01:191,0.715868,0.008,21.63
1,FQKVITEY,HLA-B15,HLA-B15:74,0.702537,0.433,24.99
2,VMADRTRHL,HLA-E01,HLA-E01:01,0.384711,0.032,778.44
3,ANADLEVKI,HLA-C06,HLA-C06:03,0.211832,4.290,5053.32


In [7]:
#some superfamilies are known

data = {
    "LIDGIFLRY": None,          #ref - hla a0101  
    "VMADRTRHL": "HLA-E01",      #ref - e0103
    "ANADLEVKI": "HLA-C06",       #ref - HLA-C*0602
                    
}

df = pd.DataFrame(list(data.items()), columns=['epitope', 'family'])

df = predictor.pan_hla_matching_prediction(df, 'epitope', 'family')
display(df)


,epitope,family,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,NaN,SLA-1-YDL01,0.733159,0.009,17.94
1,VMADRTRHL,HLA-E01,HLA-E01:01,0.384711,0.032,778.44
2,ANADLEVKI,HLA-C06,HLA-C06:03,0.211832,4.290,5053.32


###  **Testing final prediction function**

In [4]:
from src.run_netmhcpan import run_netmhcpan

In [8]:
df = pd.read_csv('../datasets/chowell.txt', sep='\t')[:100]
run_netmhcpan('single', PATH_TO_NETMHCPAN, df = df, epitope_colname='antigen.epitope', hla_column= 'hla')

,antigen.epitope,hla,immunogenicity,matched_hla,%Rank Score_BA,%Rank_BA,Aff(nM)
0,ITDFNIDTY,HLA-A*01:01,Positive,HLA-A01:01,0.732399,0.008,18.09
1,ATDALMTGF,HLA-A*01:01,Positive,HLA-A01:01,0.427019,0.133,492.51
2,SSNIMSESY,HLA-A*01:01,Positive,HLA-A01:01,0.464926,0.095,326.81
3,VSEKYTDMY,HLA-A*01:01,Positive,HLA-A01:01,0.710627,0.009,22.90
4,FTDWANKQY,HLA-A*01:01,Positive,HLA-A01:01,0.715700,0.009,21.67
...,...,...,...,...,...,...,...
95,IADMGHLKY,HLA-A*01:01,Negative,HLA-A01:01,0.676161,0.011,33.24
96,DSDLRNDSY,HLA-A*01:01,Negative,HLA-A01:01,0.556244,0.040,121.67
97,ITEDKTELY,HLA-A*01:01,Negative,HLA-A01:01,0.718329,0.009,21.07
98,ALQEALTEHY,HLA-A*01:01,Negative,HLA-A01:01,0.259031,0.589,3032.42


In [6]:
data = {
    "LIDGIFLRY": None,          #ref - hla a0101  
    "VXADRTRHL": "HLA-E01",      #ref - e0103
    "ANADLEVKI": "HLA-C06",       #ref - HLA-C*0602
                    
}

df = pd.DataFrame(list(data.items()), columns=['epitope', 'family'])

run_netmhcpan('pan', PATH_TO_NETMHCPAN, df, epitope_colname='epitope', species = 'mmu',supergroup_column='family')

,epitope,family,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
0,LIDGIFLRY,NaN,H-2-Dq,0.142449,9.208,10705.50
1,VXADRTRHL,HLA-E01,H-2-Kb,0.316589,2.090,1626.78
2,ANADLEVKI,HLA-C06,H-2-Kq,0.220784,5.759,4586.85


###  **Performance evaluation**

 1. **Epitope-hla pairs as input**

In [6]:
df = pd.read_csv('../datasets/chowell.txt', sep='\t')
start = time.time()
run_netmhcpan('single', PATH_TO_NETMHCPAN, df = df, epitope_colname='antigen.epitope', hla_column= 'hla')


,antigen.epitope,hla,immunogenicity,matched_hla,%Rank Score_BA,%Rank_BA,Aff(nM)
0,ITDFNIDTY,HLA-A*01:01,Positive,HLA-A01:01,0.732399,0.008,18.09
1,ATDALMTGF,HLA-A*01:01,Positive,HLA-A01:01,0.427019,0.133,492.51
2,SSNIMSESY,HLA-A*01:01,Positive,HLA-A01:01,0.464926,0.095,326.81
3,VSEKYTDMY,HLA-A*01:01,Positive,HLA-A01:01,0.710627,0.009,22.90
4,FTDWANKQY,HLA-A*01:01,Positive,HLA-A01:01,0.715700,0.009,21.67
...,...,...,...,...,...,...,...
7577,VMAPRTLVL,HLA-E*01033,Positive,NaN,NaN,NaN,NaN
7578,VMAPRTLLL,HLA-E*01033,Positive,NaN,NaN,NaN,NaN
7579,VMAPRALLL,HLA-E*01033,Positive,NaN,NaN,NaN,NaN
7580,VMAPRTLFL,HLA-E*01033,Positive,NaN,NaN,NaN,NaN


In [7]:
res = time.time() - start
print(f'{len(df)} pairs, {round(res,2)} sec total')
print(f'1 pair - approx {round(res/len(df),4)} sec')

7582 pairs, 67.92 sec total
1 pair - approx 0.009 sec


**2. Species-agnostic pan-prediction**

In [8]:
df = pd.DataFrame(pd.read_csv('../datasets/chowell.txt', sep='\t')[65:75]['antigen.epitope'])
start = time.time()
run_netmhcpan('pan', PATH_TO_NETMHCPAN, df, epitope_colname='antigen.epitope', species = None)


,antigen.epitope,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
65,FTDAGFEHY,HLA-A01:01,0.794687,0.006,9.22
66,ETEREYFNRY,HLA-A01:284,0.574794,0.051,99.55
67,ATDLTREVY,HLA-A36:01,0.670490,0.015,35.35
68,DTDEYVLKY,HLA-A01:284,0.728170,0.009,18.94
69,FAEHNDLQY,HLA-A01:126,0.624172,0.018,58.35
70,DTEKELQALY,HLA-A01:71,0.524782,0.033,171.02
71,ISDAAQLPHDY,HLA-A36:04,0.626611,0.025,56.83
72,ETDPDAQAKAY,HLA-A01:284,0.457106,0.155,355.67
73,IVEGENHTY,SLA-1-YDL01,0.610848,0.038,67.39
74,IIDHHDNTY,SLA-1-YDL01,0.731991,0.009,18.17


In [9]:
res = time.time() - start
print(f'{len(df)} epitopes, {round(res,2)} sec total')
print(f'1 epitope - approx {round(res/len(df),4)} sec')

10 epitopes, 82.37 sec total
1 epitope - approx 8.2373 sec


**3. Species-restricted pan-prediction - hs**

In [10]:
df = pd.DataFrame(pd.read_csv('../datasets/chowell.txt', sep='\t')[300:330]['antigen.epitope'])
start = time.time()
run_netmhcpan('pan', PATH_TO_NETMHCPAN, df, epitope_colname='antigen.epitope', species = 'hs')


,antigen.epitope,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
300,RSDRREDYY,HLA-A36:01,0.717975,0.009,21.15
301,NLDSQRLQY,HLA-A01:191,0.617517,0.014,62.70
302,TLDDESLKY,HLA-A01:191,0.663877,0.009,37.97
303,RTENPKHSQY,HLA-A36:01,0.585185,0.043,88.96
304,KTDDLTMVLY,HLA-A36:01,0.780420,0.007,10.76
305,TTDNEISRTEY,HLA-A01:284,0.573253,0.052,101.22
306,LSDIVIEKY,HLA-A01:126,0.648125,0.013,45.02
307,LLEDKHFQSY,HLA-A36:01,0.514993,0.101,190.12
308,YTETEPYHNY,HLA-A01:01,0.705772,0.009,24.13
309,QLDSAVKNLY,HLA-A36:01,0.525107,0.089,170.42


In [11]:
res = time.time() - start
print(f'{len(df)} epitopes, {round(res,2)} sec total')
print(f'1 epitope - approx {round(res/len(df),4)} sec')

30 epitopes, 122.47 sec total
1 epitope - approx 4.0825 sec


**3. Species-restricted pan-prediction - mmu**


In [13]:
start = time.time()
run_netmhcpan('pan', PATH_TO_NETMHCPAN, df, epitope_colname='antigen.epitope', species = 'mmu')


,antigen.epitope,Allele,%Rank Score_BA,%Rank_BA,Aff(nM)
300,RSDRREDYY,H-2-Db,0.068242,10.831,23894.68
301,NLDSQRLQY,H-2-Kq,0.154714,14.922,9375.04
302,TLDDESLKY,H-2-Kq,0.125437,22.624,12869.06
303,RTENPKHSQY,H-2-Dd,0.027072,31.045,37304.20
304,KTDDLTMVLY,H-2-Kq,0.143041,17.608,10637.13
305,TTDNEISRTEY,H-2-Dq,0.047985,38.878,29750.33
306,LSDIVIEKY,H-2-Dq,0.104705,15.586,16105.13
307,LLEDKHFQSY,H-2-Dq,0.107919,14.891,15554.74
308,YTETEPYHNY,H-2-Kq,0.160357,13.794,8819.76
309,QLDSAVKNLY,H-2-Dq,0.098645,16.975,17196.57


In [14]:
res = time.time() - start
print(f'{len(df)} epitopes, {round(res,2)} sec total')
print(f'1 epitope - approx {round(res/len(df),4)} sec')

30 epitopes, 5.32 sec total
1 epitope - approx 0.1774 sec
